In [33]:
import sys, os
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv(dotenv_path=Path('../.env'), override=False)

PROJECT_ROOT = Path('..').resolve()
for p in [str(PROJECT_ROOT), str(PROJECT_ROOT / 'src')]:
    if p not in sys.path:
        sys.path.insert(0, p)

DATA_DIR = PROJECT_ROOT / 'data'
CSV_PATH  = DATA_DIR / 'structured' / 'daily_sales.csv'
TEXT_DIR  = DATA_DIR / 'unstructured'

assert DATA_DIR.exists(), f'data/ not found at {DATA_DIR}'
assert CSV_PATH.exists(),  f'daily_sales.csv not found at {CSV_PATH}'
assert TEXT_DIR.exists(),  f'unstructured/ not found at {TEXT_DIR}'

print(f'CSV  : {CSV_PATH}')
print(f'Texts: {len(list(TEXT_DIR.glob("*_product_page.txt")))} product pages')

from part1.llm import LLMClient
_llm = LLMClient()
print(f'LLM  : {_llm.info}')


CSV  : /Users/CuiCuiLee/Desktop/DSAN-6725/spring-2026-a03-xinzhouli00/data/structured/daily_sales.csv
Texts: 10 product pages
LLM  : groq / llama-3.3-70b-versatile


In [32]:
from part2.pipeline import classify_query, TEST_QUESTIONS

labels = {1:'CSV Only', 2:'CSV Only', 3:'Text Only',
          4:'Text Only', 5:'Both', 6:'Both'}

print(f'{"#":<3} {"Expected":<12} {"Got":<25} Question')
print('-' * 85)
for i, q in enumerate(TEST_QUESTIONS, 1):
    tags = classify_query(q)
    print(f'Q{i:<2} {labels[i]:<10}   {str(tags):<25} {q[:42]}')


#   Expected     Got                       Question
-------------------------------------------------------------------------------------
Q1  CSV Only     ['csv']                   What was the total revenue for Electronics
Q2  CSV Only     ['csv']                   Which region had the highest sales volume?
Q3  Text Only    ['text']                  What are the key features of the Wireless 
Q4  Text Only    ['text']                  What do customers say about the Air Fryer'
Q5  Both         ['csv', 'text']           Which product has the best customer review
Q6  Both         ['csv', 'text']           I want a product for fitness that is highl


In [34]:
from part2.pipeline import answer_question

def run_q(n: int):
    q = TEST_QUESTIONS[n - 1]
    print(f'\n{"═"*72}')
    print(f'Q{n}: {q}')
    print('─' * 72)
    result = answer_question(DATA_DIR, q)
    print(f'Sources  : {result.tags}')
    print()
    print(result.answer)
    return result


In [35]:
r1 = run_q(1)



════════════════════════════════════════════════════════════════════════
Q1: What was the total revenue for Electronics category in December 2024?
────────────────────────────────────────────────────────────────────────
Sources  : ['csv']

According to the "December 2024 Revenue by Category" table, the total revenue for the Electronics category in December 2024 was $142,864.31.


In [36]:
r2 = run_q(2)



════════════════════════════════════════════════════════════════════════
Q2: Which region had the highest sales volume?
────────────────────────────────────────────────────────────────────────
Sources  : ['csv']

According to the "Units Sold by Region" table, the Central region had the highest sales volume with 6,779 units sold. Additionally, the `awk units_by_region` output confirms this, showing that the Central region had the highest units sold at 13,758.5.


In [37]:
r3 = run_q(3)



════════════════════════════════════════════════════════════════════════
Q3: What are the key features of the Wireless Bluetooth Headphones?
────────────────────────────────────────────────────────────────────────
Sources  : ['text']

The key features of the Wireless Bluetooth Headphones (ELEC001) are:

1. Active Noise Cancellation (ANC) with transparency mode
2. Bluetooth 5.2 for stable connectivity up to 30ft range
3. 40-hour battery life, quick charge (10 min = 3 hours playback)
4. Premium memory foam ear cushions for all-day comfort
5. Foldable design with carrying case included
6. Built-in microphone for calls and voice assistant

These features are mentioned in the ELEC001 product page (ELEC001_product_page.txt).


In [38]:
r4 = run_q(4)



════════════════════════════════════════════════════════════════════════
Q4: What do customers say about the Air Fryer's ease of cleaning?
────────────────────────────────────────────────────────────────────────
Sources  : ['text']

According to the customer reviews on the HOME003 product page, customers have mentioned that the Air Fryer is "Easy to clean" (Review 1 - Patricia W.) and that "cleanup is a breeze" (Review 3 - Lisa G.). This suggests that customers find the Air Fryer to be relatively easy to clean.


In [39]:
r5 = run_q(5)



════════════════════════════════════════════════════════════════════════
Q5: Which product has the best customer reviews and how well is it selling?
────────────────────────────────────────────────────────────────────────
Sources  : ['csv', 'text']

Based on the provided data, the product with the best customer reviews is the dog food with an average rating of 4.5/5 (6,432 reviews) from the file "PETS001_product_page.txt". However, another product, the coffee from "FOOD001_product_page.txt", has an average rating of 4.6/5 (4,123 reviews), which is even higher.

To determine how well these products are selling, we can refer to the "Top 15 Products by Units Sold" table. The "Dog Food Premium 10kg" is ranked 12th with 886 units sold. Unfortunately, the coffee product is not listed in the top 15 products by units sold, so we cannot directly compare its sales performance to the other products.

Therefore, based on the available data, the product with the best customer reviews that is also 

In [40]:
r6 = run_q(6)



════════════════════════════════════════════════════════════════════════
Q6: I want a product for fitness that is highly rated and sells well in the West region. What do you recommend?
────────────────────────────────────────────────────────────────────────
Sources  : ['csv', 'text']

Based on the provided data, I recommend the "Running Shoes Men" product for fitness that is highly rated and sells well in the West region. 

According to the "Top 10 Products in West Region (units sold)" table, "Running Shoes Men" has sold 301 units in the West region. Additionally, in the "Top 15 Products by Units Sold" table, "Running Shoes Men" has sold 1324 units overall, indicating its popularity.

Although there are no specific customer reviews provided for "Running Shoes Men", its high sales figures suggest that it is a well-liked product. If you'd like to consider other products with available customer reviews, the "Python Programming Guide" book has an average rating of 4.6/5 (2,234 reviews) an

In [41]:
results = [r1, r2, r3, r4, r5, r6]
sections = []
for i, res in enumerate(results, 1):
    sections.append(f'# Question {i}\n{res.format_for_file()}')

out = PROJECT_ROOT / 'part2_results.txt'
out.write_text('\n\n---\n\n'.join(sections), encoding='utf-8')
print(f'Saved {len(sections)} answers → {out}')


Saved 6 answers → /Users/CuiCuiLee/Desktop/DSAN-6725/spring-2026-a03-xinzhouli00/part2_results.txt
